# PyTorch Transformer-Only Practice Workbook

[Open in Colab](https://colab.research.google.com/github/mudit1729/mlwiki-colabs/blob/main/pytorch-code-samples/pytorch-transformer-implementations.ipynb)

Transformer-only implementation drills for scaled dot-product attention, multi-head attention, positional encoding, RoPE, transformer encoder/decoder blocks, and Vision Transformers.

Source wiki page: [transformer implementation notes](https://auto-driving-wiki.up.railway.app/page/wiki/syntheses/pytorch-code-samples/paper-implementations-transformers)

This workbook is **PyTorch-only** and intentionally does **not** include completed answers. It gives you boilerplate signatures plus basic test helpers. Fill the TODOs first, then uncomment each `run_exercise_XX_basic_tests()` call.


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__)
print("device", device)


def assert_shape(x, shape):
    assert tuple(x.shape) == tuple(shape), f"expected shape {shape}, got {tuple(x.shape)}"


def assert_finite_tensor(x):
    assert torch.isfinite(x).all(), "tensor contains non-finite values"


def assert_close(a, b, atol=1e-5, rtol=1e-5):
    torch.testing.assert_close(a, b, atol=atol, rtol=rtol)


# Workbook setup smoke test.
_x = torch.randn(2, 3, device=device)
assert_shape(_x, (2, 3))
assert_finite_tensor(_x)


## Practice Flow

1. Read the exercise prompt and shape hints.
2. Fill the TODO implementation body.
3. Uncomment `run_exercise_XX_basic_tests()`.
4. Run the cell and fix the implementation until the basic tests pass.
5. Explain tensor shapes out loud before writing each attention reshape.

Keep this workbook answer-free while practicing. Add your own edge-case tests after the basic helper passes.


## Exercise 01: Scaled Dot-Product Attention

Implement attention from first principles.

Expected shapes:

- `q`: `(batch, heads, query_len, head_dim)`
- `k`: `(batch, heads, key_len, head_dim)`
- `v`: `(batch, heads, key_len, head_dim)`
- `mask`: broadcastable to `(batch, heads, query_len, key_len)`; `True` means keep and `False` means block
- output: `(batch, heads, query_len, head_dim)`

Concepts tested: scale by `sqrt(head_dim)`, mask before softmax, causal masking, and attention probabilities.


In [ ]:
# Exercise 01: Scaled Dot-Product Attention
# Fill the TODOs, then run the basic test helper at the bottom of this cell.

def scaled_dot_product_attention(q, k, v, mask=None, dropout_p=0.0, is_causal=False):
    """Return (output, attention_weights).

    q, k, v are shaped (B, H, Tq/Tk, Dh). mask uses True=keep, False=block.
    """
    # TODO: compute scores = q @ k.transpose(-2, -1) / sqrt(Dh).
    # TODO: apply optional causal mask for query/key lengths.
    # TODO: apply optional boolean mask by filling blocked logits with -inf.
    # TODO: softmax over key dimension, optional dropout, then weighted sum over v.
    raise NotImplementedError("Implement scaled_dot_product_attention")


def run_exercise_01_basic_tests():
    """Run after filling the TODO implementation above."""
    torch.manual_seed(0)
    q = torch.randn(2, 3, 4, 8, device=device)
    k = torch.randn(2, 3, 5, 8, device=device)
    v = torch.randn(2, 3, 5, 8, device=device)
    out, attn = scaled_dot_product_attention(q, k, v)
    assert_shape(out, (2, 3, 4, 8))
    assert_shape(attn, (2, 3, 4, 5))
    assert_finite_tensor(out)
    assert_close(attn.sum(dim=-1), torch.ones(2, 3, 4, device=device))

    mask = torch.ones(2, 1, 4, 5, dtype=torch.bool, device=device)
    mask[..., -1] = False
    _, masked_attn = scaled_dot_product_attention(q, k, v, mask=mask)
    assert_close(masked_attn[..., -1], torch.zeros_like(masked_attn[..., -1]))
    print("exercise 01 basic tests passed")

# Uncomment after implementing the exercise:
# run_exercise_01_basic_tests()


## Exercise 02: Multi-Head Attention

Build a reusable multi-head attention module on top of Exercise 01.

Expected shapes:

- input `x`: `(batch, seq_len, embed_dim)` for self-attention
- optional `context`: `(batch, context_len, embed_dim)` for cross-attention
- output: `(batch, seq_len, embed_dim)`

Concepts tested: Q/K/V projections, head splitting, transpose/contiguous/view discipline, output projection, and optional cross-attention.


In [ ]:
# Exercise 02: Multi-Head Attention
# Fill the TODOs, then run the basic test helper at the bottom of this cell.

class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, dropout=0.0):
        super().__init__()
        assert embed_dim % num_heads == 0, "embed_dim must divide num_heads"
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        # TODO: define q_proj, k_proj, v_proj, out_proj, and dropout.

    def _split_heads(self, x):
        """Convert (B, T, D) to (B, H, T, Dh)."""
        # TODO: reshape and transpose.
        raise NotImplementedError("Implement _split_heads")

    def _merge_heads(self, x):
        """Convert (B, H, T, Dh) to (B, T, D)."""
        # TODO: transpose, make contiguous if needed, then reshape.
        raise NotImplementedError("Implement _merge_heads")

    def forward(self, x, context=None, mask=None, is_causal=False):
        """Self-attention when context=None; cross-attention otherwise."""
        # TODO: project Q from x; project K/V from context if provided else x.
        # TODO: split heads, call scaled_dot_product_attention, merge heads, apply out_proj.
        raise NotImplementedError("Implement forward")


def run_exercise_02_basic_tests():
    """Run after filling the TODO implementation above."""
    torch.manual_seed(0)
    mha = MultiHeadAttention(embed_dim=32, num_heads=4).to(device)
    x = torch.randn(2, 6, 32, device=device)
    out = mha(x)
    assert_shape(out, (2, 6, 32))
    assert_finite_tensor(out)

    memory = torch.randn(2, 9, 32, device=device)
    cross = mha(x, context=memory)
    assert_shape(cross, (2, 6, 32))
    assert_finite_tensor(cross)
    print("exercise 02 basic tests passed")

# Uncomment after implementing the exercise:
# run_exercise_02_basic_tests()


## Exercise 03: Sinusoidal Positional Encoding

Implement the original transformer sinusoidal positional encoding table.

Expected shape: `(max_len, d_model)`.

Concepts tested: even/odd dimensions, exponential frequency schedule, broadcasting, and device/dtype handling.


In [ ]:
# Exercise 03: Sinusoidal Positional Encoding
# Fill the TODOs, then run the basic test helper at the bottom of this cell.

def sinusoidal_positional_encoding(max_len, d_model, device=None, dtype=torch.float32):
    """Return a (max_len, d_model) sinusoidal positional encoding table."""
    # TODO: create positions shaped (max_len, 1).
    # TODO: create div terms for even dimensions.
    # TODO: fill sin on even indices and cos on odd indices.
    # TODO: handle odd d_model without shape errors.
    raise NotImplementedError("Implement sinusoidal_positional_encoding")


def run_exercise_03_basic_tests():
    """Run after filling the TODO implementation above."""
    pe = sinusoidal_positional_encoding(8, 6, device=device)
    assert_shape(pe, (8, 6))
    assert_finite_tensor(pe)
    assert_close(pe[0, 0::2], torch.zeros(3, device=device))
    assert_close(pe[0, 1::2], torch.ones(3, device=device))

    pe_odd = sinusoidal_positional_encoding(4, 5, device=device)
    assert_shape(pe_odd, (4, 5))
    assert_finite_tensor(pe_odd)
    print("exercise 03 basic tests passed")

# Uncomment after implementing the exercise:
# run_exercise_03_basic_tests()


## Exercise 04: RoPE Rotary Position Embeddings

Implement a RoPE cache and apply rotary embeddings to query/key tensors.

Expected shapes:

- input `x`: `(batch, heads, seq_len, head_dim)`
- `head_dim` must be even
- `cos`, `sin`: broadcastable to `(1, 1, seq_len, head_dim)`
- output: same shape as `x`

Concepts tested: pairwise feature rotation, norm preservation, broadcasting, and LLM-style positional encoding.


In [ ]:
# Exercise 04: RoPE Rotary Position Embeddings
# Fill the TODOs, then run the basic test helper at the bottom of this cell.

def build_rope_cache(seq_len, head_dim, base=10000, device=None, dtype=torch.float32):
    """Return cos, sin caches broadcastable to (B, H, T, Dh)."""
    assert head_dim % 2 == 0, "head_dim must be even for RoPE"
    # TODO: compute inverse frequencies for head_dim//2 pairs.
    # TODO: build angles for each position and frequency.
    # TODO: duplicate/interleave angles so cos/sin match full head_dim.
    # TODO: return cos and sin shaped (1, 1, seq_len, head_dim).
    raise NotImplementedError("Implement build_rope_cache")


def rotate_half(x):
    """Rotate feature pairs: (x0, x1) -> (-x1, x0)."""
    # TODO: split x into even and odd features, rotate pairs, then restore original layout.
    raise NotImplementedError("Implement rotate_half")


def apply_rope(x, cos, sin):
    """Apply RoPE to x using broadcastable cos/sin caches."""
    # TODO: return x * cos + rotate_half(x) * sin.
    raise NotImplementedError("Implement apply_rope")


def run_exercise_04_basic_tests():
    """Run after filling the TODO implementations above."""
    torch.manual_seed(0)
    x = torch.randn(2, 4, 7, 8, device=device)
    cos, sin = build_rope_cache(seq_len=7, head_dim=8, device=device)
    y = apply_rope(x, cos, sin)
    assert_shape(y, x.shape)
    assert_finite_tensor(y)
    assert_close(y.norm(dim=-1), x.norm(dim=-1), atol=1e-5, rtol=1e-5)
    print("exercise 04 basic tests passed")

# Uncomment after implementing the exercise:
# run_exercise_04_basic_tests()


## Exercise 05: Transformer Encoder Layer

Implement a pre-norm transformer encoder block.

Expected shape: `(batch, seq_len, embed_dim)` in and out.

Concepts tested: residual connections, layer norm placement, self-attention, feed-forward expansion, dropout, and source masks.


In [ ]:
# Exercise 05: Transformer Encoder Layer
# Fill the TODOs, then run the basic test helper at the bottom of this cell.

class TransformerEncoderLayer(nn.Module):
    def __init__(self, embed_dim, num_heads, mlp_ratio=4, dropout=0.0):
        super().__init__()
        # TODO: define norm1, self_attn, norm2, feed-forward MLP, and dropout.

    def forward(self, x, src_mask=None):
        """Pre-norm encoder: x + self_attn(norm1(x)), then x + mlp(norm2(x))."""
        # TODO: implement residual self-attention block.
        # TODO: implement residual MLP block.
        raise NotImplementedError("Implement TransformerEncoderLayer.forward")


def run_exercise_05_basic_tests():
    """Run after filling the TODO implementation above."""
    torch.manual_seed(0)
    layer = TransformerEncoderLayer(embed_dim=32, num_heads=4, dropout=0.0).to(device)
    x = torch.randn(2, 6, 32, device=device)
    out = layer(x)
    assert_shape(out, (2, 6, 32))
    assert_finite_tensor(out)
    print("exercise 05 basic tests passed")

# Uncomment after implementing the exercise:
# run_exercise_05_basic_tests()


## Exercise 06: Transformer Decoder Layer

Implement a pre-norm transformer decoder block with causal self-attention and encoder-decoder cross-attention.

Expected shapes:

- `tgt`: `(batch, target_len, embed_dim)`
- `memory`: `(batch, source_len, embed_dim)`
- output: `(batch, target_len, embed_dim)`

Concepts tested: causal masks, self-attention, cross-attention, residual ordering, and MLP block reuse.


In [ ]:
# Exercise 06: Transformer Decoder Layer
# Fill the TODOs, then run the basic test helper at the bottom of this cell.

def make_causal_mask(seq_len, device=None):
    """Return a boolean mask shaped (1, 1, seq_len, seq_len), True=keep."""
    # TODO: return lower-triangular keep mask.
    raise NotImplementedError("Implement make_causal_mask")


class TransformerDecoderLayer(nn.Module):
    def __init__(self, embed_dim, num_heads, mlp_ratio=4, dropout=0.0):
        super().__init__()
        # TODO: define norm layers, self_attn, cross_attn, MLP, and dropout.

    def forward(self, tgt, memory, tgt_mask=None, memory_mask=None):
        """Pre-norm decoder with causal self-attention and cross-attention."""
        # TODO: apply causal self-attention to tgt.
        # TODO: apply cross-attention from tgt to memory.
        # TODO: apply residual MLP.
        raise NotImplementedError("Implement TransformerDecoderLayer.forward")


def run_exercise_06_basic_tests():
    """Run after filling the TODO implementations above."""
    mask = make_causal_mask(5, device=device)
    assert_shape(mask, (1, 1, 5, 5))
    assert mask.dtype == torch.bool
    assert mask[0, 0, 4, 0].item() is True
    assert mask[0, 0, 0, 4].item() is False

    torch.manual_seed(0)
    layer = TransformerDecoderLayer(embed_dim=32, num_heads=4, dropout=0.0).to(device)
    tgt = torch.randn(2, 5, 32, device=device)
    memory = torch.randn(2, 7, 32, device=device)
    out = layer(tgt, memory)
    assert_shape(out, (2, 5, 32))
    assert_finite_tensor(out)
    print("exercise 06 basic tests passed")

# Uncomment after implementing the exercise:
# run_exercise_06_basic_tests()


## Exercise 07: Vision Transformer Patch Embedding And Tiny ViT

Implement patch embedding and a minimal ViT classifier using your encoder layer.

Expected shapes:

- image batch: `(batch, channels, height, width)`
- patch tokens: `(batch, num_patches, embed_dim)`
- logits: `(batch, num_classes)`

Concepts tested: patchify with `Conv2d`, class token, positional embedding, encoder stack, and classification head.


In [ ]:
# Exercise 07: Vision Transformer Patch Embedding And Tiny ViT
# Fill the TODOs, then run the basic test helper at the bottom of this cell.

class ViTPatchEmbedding(nn.Module):
    def __init__(self, image_size, patch_size, in_channels, embed_dim):
        super().__init__()
        assert image_size % patch_size == 0, "image_size must divide patch_size"
        self.image_size = image_size
        self.patch_size = patch_size
        self.num_patches = (image_size // patch_size) ** 2
        # TODO: define a Conv2d projection with kernel_size=stride=patch_size.

    def forward(self, x):
        """Return patch tokens shaped (B, num_patches, embed_dim)."""
        # TODO: project image to patch grid, flatten spatial dimensions, transpose to tokens.
        raise NotImplementedError("Implement ViTPatchEmbedding.forward")


class TinyViTClassifier(nn.Module):
    def __init__(self, image_size=16, patch_size=4, in_channels=3, embed_dim=32, num_heads=4, depth=2, num_classes=10):
        super().__init__()
        # TODO: define patch_embed, cls_token, pos_embed, encoder layers, norm, and head.

    def forward(self, x):
        # TODO: create patch tokens, prepend class token, add positional embeddings.
        # TODO: pass through encoder layers and classify the final class token.
        raise NotImplementedError("Implement TinyViTClassifier.forward")


def run_exercise_07_basic_tests():
    """Run after filling the TODO implementations above."""
    torch.manual_seed(0)
    patch = ViTPatchEmbedding(image_size=16, patch_size=4, in_channels=3, embed_dim=32).to(device)
    images = torch.randn(2, 3, 16, 16, device=device)
    tokens = patch(images)
    assert_shape(tokens, (2, 16, 32))
    assert_finite_tensor(tokens)

    model = TinyViTClassifier(image_size=16, patch_size=4, in_channels=3, embed_dim=32, num_heads=4, depth=2, num_classes=5).to(device)
    logits = model(images)
    assert_shape(logits, (2, 5))
    assert_finite_tensor(logits)
    print("exercise 07 basic tests passed")

# Uncomment after implementing the exercise:
# run_exercise_07_basic_tests()


## After You Finish

Use the source wiki page to compare your implementation against the expected architecture details:

- [[wiki/syntheses/pytorch-code-samples/paper-implementations-transformers]]
- [[wiki/syntheses/pytorch-code-samples/hard]]
- [[wiki/syntheses/pytorch-code-samples/cheatsheet]]

Follow-up drills after the basics pass:

1. Add key/value caching to decoder self-attention.
2. Add attention dropout and residual dropout tests.
3. Add RoPE to multi-head attention by rotating Q/K before attention.
4. Add learned absolute positional embeddings to the ViT exercise.
5. Replace the manual MHA with `torch.nn.functional.scaled_dot_product_attention` and compare outputs.
